In [1]:
import time
from urls import urls

import re
import json

from pymongo import MongoClient
import os
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

from typing import Literal, Dict
from typing import Literal, Dict, Optional
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


load_dotenv()

True

In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client["insurance_db"]
collection = db["health_insurance"]

In [3]:
mixtral = 'mixtral-8x7b-32768'
llama = 'llama3-70b-8192'

#model = ChatGroq(temperature=0, model_name=llama) 

model = ChatAnthropic(temperature=0.4, model_name='claude-3-5-sonnet-20240620') 

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert pymongo code generator that generates code for given question by the user. DONT again instatiate client and collection in the output. I DONT NEED ANY VERBOSE IN THE OUTPUT. JUST NEED CODE
Below is the schema of mongo db table where provider is name of health insurance company and insurance_name is name of health insurance policy.
In below schema there are 10 features from copayment to no claim bonus. For a insurance policy below schema tells whether the feature in this policy is good/bad/neutral/NA.
Like wise you have lot of data related to lot of providers. policy name will be in lower case seperated by -
    
MONGO DB details:
client = MongoClient("mongodb://localhost:27017/")
db = client["insurance_db"]
collection = db["health_insurance"]
     
SCHEMA:
{{'insurance_name': 'united-india-individual-gold-plan',
  'provider': 'united-india',
  'description': 'United India Individual Gold Plan is a comprehensive health insurance policy that covers various medical expenses, including hospitalization, day care treatments, and alternative medicine.',
  'features': {{
      'co-payment': {{'rating': 'good', 'details': 'No co-payment, the insurer will bear the entire cost of treatment up to the sum insured.'}},
      'Room-rent-limit': {{'rating': 'bad', 'details': 'Room rent limit of 1% of the sum insured for normal rooms and 2% for ICU.'}},
      'Disease-sub-limit': {{'rating': 'bad', 'details': 'Disease-wise sub-limits for certain diseases like Cataracts, Hernia, Hysterectomy, etc.'}},
      'Pre-Post-hospitalization': {{'rating': 'good', 'details': 'Pre and post hospitalization expenses covered up to 10% of sum insured for 30 days before and 60 days after discharge.'}},
      'critical-illness': {{'rating': 'NA', 'details': 'NA'}},
      'Low-Waiting-period': {{'rating': 'neutral', 'details': 'Reasonable waiting period of 3 years for pre-existing diseases.'}},
      'Daycare-Treatment-cover': {{'rating': 'good', 'details': 'Day care treatments covered, including minor procedures like dialysis, chemotherapy, and minor surgeries.'}},
      'Restoration-Benefit': {{'rating': 'bad', 'details': 'No restoration benefit, the policy does not restore the cover after a claim is made.'}},
      'maternity-coverage': {{'rating': 'NA', 'details': 'NA'}},
      'no-claim-bonus': {{'rating': 'bad', 'details': 'No bonus for being healthy and not claiming insurance.'}}
  }}
}}


"""),
    ("human", '''
     RULES FOR CODE GENERATION:
     1) Below is the list of insurance providers as stored in mongo db. Given any question related to any provider, PLEASE MAP to below names. This is MANDATORY.
     acko, aditya-birla, bajaj-allianz, bharti-axa, care, hdfc-ergo, icici-lombard, iffco-tokio, manipal-cigna, new-india-assurance, niva-bupa-erstwhile-max-bupa, oriental-insurance, reliance-general, royal-sundaram, sbi, star-health, tata-aig, united-india
     2) whenever a question is asked about a certain policy under a provider, remember policy name will be in lower case where multiple words are seperated by -.
     3) Always dont do a direct match of policy name rather check for partial match ex- LIKE in SQL. Always include insurance name in the output. I need ALL the matching documents.
     4) I will be using exec command to execute your output, so please print results in your code.
     5) When filtering fields in MongoDB, DO NOT use Python dictionary comprehensions directly in the projection parameter. MongoDB projections cannot process Python expressions like "{{k: v for k, v in "$features".items()}}".
     6) To filter subdocuments or embedded objects (like features with specific ratings), either:
            - Use the appropriate MongoDB operators in the query
            - OR retrieve the complete field and filter it in Python after getting the results
     7) I need ONLY pymongo CODE as output WITHOUT any verbose and WITHOUT any other explanation and i DONT even need, here is your code start text.

{question}
     ''')
])

# Fix the RunnableMap implementation
chain = (
    RunnableMap({
        "question": lambda x: x["question"]
    })
    | template 
    | model 
    | StrOutputParser()
)

In [5]:
queries = ['What policies are available under hdfc ergo plan',
           'What is bad in icici lombard ihealth plan',
           'List me good and neutral features of hdfc ergo optima secure policy']

In [12]:
queries = ['For all the policies under aditya birla provider, i want to know if there is good room rent limit']

In [6]:
final_code = []
cnt=0
for q in queries:
    response = chain.invoke({"question": q}).replace('```', '')
    final_code.append(response)
    print(cnt)
    cnt+=1

0


In [13]:
response = chain.invoke({"question": queries[0]}).replace('```', '')

In [14]:
print(response)

query = {"provider": "aditya-birla", "features.Room-rent-limit.rating": "good"}
projection = {"insurance_name": 1, "features.Room-rent-limit": 1, "_id": 0}

results = collection.find(query, projection)

for result in results:
    print(f"Insurance Name: {result['insurance_name']}")
    print(f"Room Rent Limit: {result['features']['Room-rent-limit']['details']}")
    print()


In [15]:
from contextlib import redirect_stdout
import io

f = io.StringIO()
with redirect_stdout(f):
        exec(response)

In [16]:
output = f.getvalue() 

In [17]:
print(output)

Insurance Name: aditya-birla-activ-one-vip
Room Rent Limit: The insurer won’t nitpick on your choice of room since the policy has no restrictions on room rent.

Insurance Name: aditya-birla-activ-one-savr
Room Rent Limit: The policy has no restrictions on room rent, allowing you to choose any room you like.

Insurance Name: aditya-birla-activ-one-vytl
Room Rent Limit: The policy has no restrictions on room rent, allowing you to choose any room you like.

Insurance Name: aditya-birla-activ-one-nxt
Room Rent Limit: This policy has no restrictions on room rent, allowing you to choose any room you like.

Insurance Name: aditya-birla-activ-fit-preferred
Room Rent Limit: The policy has no restrictions on room rent.




In [38]:
q = 'Whats bad in bharti axa smart super health policy'

response = chain.invoke({"question": q}).replace('```', '')

In [39]:
exec(response)

Policy: bharti-axa-smart-super-health-assure
- Low-Waiting-period: Long waiting period for pre-existing diseases, 4 years.
- maternity-coverage: Maternity and Newborn Waiting Period of 9 months.
Policy: bharti-axa-smart-super-health
- Low-Waiting-period: Long waiting period for pre-existing diseases - 4 long years.
